[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/notebooks/ch02_bandits.ipynb)

<div style="background: linear-gradient(135deg, #0d1b2a 0%, #1b2838 50%, #2a3f5f 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffd700; font-size: 2.2em; margin: 0 0 10px 0;">Chapter 2: Multi-Armed Bandits</h1>
  <p style="color: #a8dadc; font-size: 1.1em; margin: 0;">Epsilon-Greedy, UCB1, and Thompson Sampling on a 10-armed Gaussian bandit — 1000 steps, 300 runs.</p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Environment Setup</strong><br>
  <span style="color: #cdd9e5;">No GPU required. Estimated runtime: ~30 seconds on a free Colab CPU.</span>
</div>

In [ ]:
!pip install numpy matplotlib --quiet

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
})
print('Setup complete.')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #ffd700;">
  <strong style="color: #ffd700;">1. The k-Armed Bandit Problem</strong>
</div>

A **k-armed bandit** is a simplified RL problem with:
- $k$ actions ("arms"), each with an unknown reward distribution.
- No state transitions — each step is independent.
- Goal: maximise cumulative reward over $T$ steps.

Each arm $a$ has a true value $q^*(a) = \mathbb{E}[R | A=a]$. The agent must **estimate** these values while **balancing exploration and exploitation**.

**Setup**: 10 arms, rewards $\sim \mathcal{N}(q^*(a), 1)$, true values $q^*(a) \sim \mathcal{N}(0, 1)$.

In [ ]:
# ── 10-Armed Gaussian Bandit ──

class BanditEnv:
    """Stationary k-armed bandit with Gaussian rewards."""

    def __init__(self, k=10, seed=None):
        self.k = k
        rng = np.random.default_rng(seed)
        self.q_star = rng.standard_normal(k)   # True action values
        self.optimal_action = int(np.argmax(self.q_star))

    def step(self, action):
        """Return noisy reward for the given arm."""
        return np.random.normal(self.q_star[action], 1.0)


# Visualise true values for one bandit instance
env0 = BanditEnv(k=10, seed=SEED)
fig, ax = plt.subplots(figsize=(10, 4))

for a in range(10):
    samples = np.random.normal(env0.q_star[a], 1.0, 200)
    ax.violinplot(samples, positions=[a], widths=0.7,
                  showmeans=True, showmedians=False)
ax.axhline(0, color='#8b949e', linestyle='--', alpha=0.5, label='Zero baseline')
ax.scatter(range(10), env0.q_star, s=80, color='#ffd700',
           zorder=5, label='True value q*(a)')
ax.set_xticks(range(10))
ax.set_xticklabels([f'Arm {i}' for i in range(10)], rotation=30)
ax.set_title('10-Armed Bandit: Reward Distributions', fontsize=13)
ax.set_ylabel('Reward')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Optimal arm: {env0.optimal_action}  (q* = {env0.q_star[env0.optimal_action]:.3f})')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #ffd700;">
  <strong style="color: #ffd700;">2. Three Bandit Algorithms</strong>
</div>

**Epsilon-Greedy**: choose the best-estimated arm most of the time, explore uniformly with probability $\varepsilon$.
$$A_t = \begin{cases} \arg\max_a Q_t(a) & \text{w.p. } 1-\varepsilon \\ \text{random arm} & \text{w.p. } \varepsilon \end{cases}$$

**UCB1** (Upper Confidence Bound): select the arm with the highest upper confidence bound, naturally trading off exploration and exploitation:
$$A_t = \arg\max_a \left[ Q_t(a) + c\sqrt{\frac{\ln t}{N_t(a)}} \right]$$

**Thompson Sampling**: maintain a Beta posterior over each arm's success probability; sample from posteriors and pick the highest sample. (We adapt for Gaussian rewards using Normal-Normal conjugacy.)

In [ ]:
# ── Algorithm implementations ──

class EpsilonGreedy:
    def __init__(self, k, epsilon=0.1):
        self.k = k
        self.epsilon = epsilon
        self.Q = np.zeros(k)   # Estimated values
        self.N = np.zeros(k)   # Pull counts

    def select_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)
        return int(np.argmax(self.Q))

    def update(self, action, reward):
        self.N[action] += 1
        # Incremental sample mean
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class UCB1:
    def __init__(self, k, c=2.0):
        self.k = k
        self.c = c
        self.Q = np.zeros(k)
        self.N = np.zeros(k)
        self.t = 0

    def select_action(self):
        self.t += 1
        # Pull each arm once at the start
        untried = np.where(self.N == 0)[0]
        if len(untried) > 0:
            return untried[0]
        ucb = self.Q + self.c * np.sqrt(np.log(self.t) / (self.N + 1e-9))
        return int(np.argmax(ucb))

    def update(self, action, reward):
        self.N[action] += 1
        self.Q[action] += (reward - self.Q[action]) / self.N[action]


class ThompsonSampling:
    """Gaussian Thompson Sampling: Normal-Normal conjugate model.
       Prior: mean ~ N(0, 1). Likelihood: R ~ N(mu, sigma^2=1).
    """
    def __init__(self, k):
        self.k = k
        self.mu = np.zeros(k)     # Posterior mean
        self.tau = np.ones(k)     # Posterior precision (inverse variance)

    def select_action(self):
        samples = np.random.normal(self.mu, 1.0 / np.sqrt(self.tau))
        return int(np.argmax(samples))

    def update(self, action, reward):
        # Update posterior: new_tau = tau + 1, new_mu = (tau*mu + reward) / new_tau
        new_tau = self.tau[action] + 1.0
        self.mu[action] = (self.tau[action] * self.mu[action] + reward) / new_tau
        self.tau[action] = new_tau


print('Algorithms defined.')

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #ffd700;">
  <strong style="color: #ffd700;">3. Running 300-Run Experiments (1000 steps each)</strong>
</div>

We average results over 300 independent bandit instances to reduce variance. For each run, a new set of true values $q^*(a)$ is sampled.

In [ ]:
# ── Experiment runner ──  (~25 sec)
K = 10
T = 1000       # Steps per run
N_RUNS = 300   # Independent bandit instances

algo_configs = [
    ('ε-Greedy (ε=0.01)', lambda: EpsilonGreedy(K, epsilon=0.01)),
    ('ε-Greedy (ε=0.10)', lambda: EpsilonGreedy(K, epsilon=0.10)),
    ('UCB1 (c=2)',        lambda: UCB1(K, c=2.0)),
    ('Thompson Sampling', lambda: ThompsonSampling(K)),
]

all_rewards   = {name: np.zeros((N_RUNS, T)) for name, _ in algo_configs}
all_optimal   = {name: np.zeros((N_RUNS, T)) for name, _ in algo_configs}

np.random.seed(SEED)
for run in range(N_RUNS):
    env = BanditEnv(k=K, seed=run)   # Fresh bandit per run
    for name, make_algo in algo_configs:
        algo = make_algo()
        for t in range(T):
            a = algo.select_action()
            r = env.step(a)
            algo.update(a, r)
            all_rewards[name][run, t] = r
            all_optimal[name][run, t] = (a == env.optimal_action)

print('Experiments complete.')

In [ ]:
# ── Plot results: average reward + % optimal action ──
COLORS = {
    'ε-Greedy (ε=0.01)': '#58a6ff',
    'ε-Greedy (ε=0.10)': '#e94560',
    'UCB1 (c=2)':        '#ffd700',
    'Thompson Sampling': '#56d364',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, _ in algo_configs:
    color = COLORS[name]
    mean_reward = all_rewards[name].mean(axis=0)
    mean_optimal = all_optimal[name].mean(axis=0) * 100
    axes[0].plot(mean_reward,  color=color, label=name, linewidth=2, alpha=0.9)
    axes[1].plot(mean_optimal, color=color, label=name, linewidth=2, alpha=0.9)

axes[0].set_title('Average Reward over Time', fontsize=13)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Average Reward')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('% Optimal Action over Time', fontsize=13)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('% Optimal Action')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('10-Armed Bandit: 300 Runs × 1000 Steps', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #ffd700;">
  <strong style="color: #ffd700;">4. Final Performance Summary</strong>
</div>

We compare the **cumulative regret** (total reward missed vs. always picking the optimal arm) and **average reward in the last 100 steps** (steady-state performance).

In [ ]:
# ── Summary stats ──
print(f"{'Algorithm':<25} {'Avg Reward (last 100)':<25} {'Optimal % (last 100)'}")
print('-' * 65)
for name, _ in algo_configs:
    avg_r = all_rewards[name][:, -100:].mean()
    opt_p = all_optimal[name][:, -100:].mean() * 100
    print(f'{name:<25} {avg_r:<25.4f} {opt_p:.1f}%')


# ── Cumulative regret plot ──
# Per-run optimal reward at each step (average q* across runs)
# We approximate optimal as the per-step mean reward of best arm
fig, ax = plt.subplots(figsize=(10, 4))

# Recalculate cumulative regret
np.random.seed(SEED)
# Estimate optimal expected reward (mean of best arm q* averaged over runs)
best_q = np.array([BanditEnv(k=K, seed=r).q_star.max() for r in range(N_RUNS)])
optimal_expected = best_q.mean()   # ~ theoretical optimum

for name, _ in algo_configs:
    color = COLORS[name]
    mean_reward = all_rewards[name].mean(axis=0)
    cum_regret = np.cumsum(optimal_expected - mean_reward)
    ax.plot(cum_regret, color=color, label=name, linewidth=2)

ax.set_title('Cumulative Regret (lower is better)', fontsize=13)
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Regret')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Chapter 2 — Key Takeaways</h3>
  <ul style="color: #c9d1d9; line-height: 1.8;">
    <li><strong>Epsilon-greedy</strong> is simple and works well; high ε explores more but exploits less at steady state.</li>
    <li><strong>UCB1</strong> provides principled optimism-in-the-face-of-uncertainty — automatically visits under-explored arms.</li>
    <li><strong>Thompson Sampling</strong> often achieves the best practical performance by sampling from posterior beliefs.</li>
    <li>All algorithms have logarithmic cumulative regret (provably optimal for UCB / Thompson Sampling).</li>
    <li>The bandit problem is a building block: RL generalises it to sequential decisions with state transitions.</li>
  </ul>
</div>